In [1]:
import qgauss as qg
import numpy as np

In [2]:
"""
NOTE: The syntax for the measurement rate is far more rudimentary compared to that of the backaction. Improvements to
this function are ongoing.

We start by modelling the familiar case of dispersive readout of a qubit coupled to a single linear mode in the vacuum. 
In this case we know that the measurement-induced dephasing will be identical to the measurement rate.
"""
# Start by defining the annihilation operator for the linear mode, and the spin-z operator for the qubit.
a = qg.tensor(qg.destroy(),qg.identity_fls(2))
sz = qg.tensor(qg.identity_cvs(),qg.sigmaz())

# Parameters
chi = 2.75; lw = 6; eps = 6

# Define the Hamiltonian and Lindbladian.
H1_disp = chi*sz*a.dag()*a + eps*a.dag() + np.conj(eps)*a
L1 = qg.lindbladian(H = H1_disp, c_ops=[np.sqrt(lw)*a])

# Calculate the measurement-induced backation.
np.real(qg.backaction_rate_steadystate(L1, qubit="e,g")[2])

np.float64(11.909604841580634)

In [3]:
"""
We can now model the measurement rate, which is defined using input-output theory and the Heisenberg-Langevin equations (HLEs), see
Gardiner & Collett, Phys. Rev. A 31, 3761 (1985). Since these equations cannot be uniquely defined from the Lindbladian and state
of the input fields, we must construct them directly from the system-bath Hamiltonian.
"""
# Define the annihilation operators for the system and the coupled environment; these need not be defined separately in this case,
#  but are done so here for clarity.
a_sys = qg.destroy()
a_bath = qg.destroy()

# Next, we defined the state of the environment, which in this case we take to be the vacuum state.
input1_bath_state = qg.vacuum()

# We now define the system-bath coupling, which in this case is a beam-splitter. The phase we have chosen conforms to the choice in
# Gardiner and Collett, which gives the following HLE for a_sys:
# d/dt a_sys = -np.sqrt(lw)*a_bath + coherent terms
# Flipping the phase on the beam-splitter would yield d/dt a_sys = np.sqrt(lw)*a_bath, etc.
H1_sysbath = 1j*np.sqrt(lw)*( (a_sys & a_bath.dag()) - (a_sys.dag() & a_bath))

# NOTE: Based on the current code, it is required that all system modes be placed first in the tensor product, and the bath modes last.

# Now, we pass these objects, along with the original dispersive Hamiltonian to the solver and notice that the measurement rate
# is identitical to the initial dephasing rate. We specify that the measurement is distinguishing between the excited and ground
# states of the qubit.
qg.measurement_rate(H1_disp, H1_sysbath, input1_bath_state, qubit='e,g')[0]

np.float64(11.909604841580638)

In [4]:
# Notice that the measurement rate includes the measurement signal and measurement noise. Since we did not specify a measurement 
# operator for the output fields, one was calculated which optimized for the largest signal. We could have also specified the 
# measurement operator, and the frequency of the output field that we would like to measure.

# Additionally, if we do not specify the pair of state to measured, the measurement rates of all possible pairs are calculated.
# Note, of course, that if the pointer states are the same (diagonal entires) then the measurement rate is zer0.

qg.measurement_rate(H1_disp, H1_sysbath, input1_bath_state)

(array([[ 0.        , 11.90960484],
        [11.90960484,  0.        ]]),
 array([[ 0.        , 11.90960484],
        [11.90960484,  0.        ]]),
 array([[0., 1.],
        [1., 0.]]))

In [6]:
"""
We now model some measurement inneficiency, where the system decays into a monitored "external" environment and an
unmonitored "internal" environment. This is handled by coupling the system mode to two environmental modes.
"""
# Define the system and bath operators
a_sys = qg.destroy()
a_ext = qg.tensor(qg.destroy(),qg.identity_cvs())
a_int = qg.tensor(qg.identity_cvs(),qg.destroy())

# Specifiy the bath state, which in this case is a low-occupancy thermal state
input2A_bath_state = qg.thermal([0.003,0.003])

# Of the total decay rate of the system mode, 95% is into the monitored channel, while 5% represents internal losses.
lw_ext = 0.95*lw; lw_int = 0.05*lw

# We now defined the system-bath Hamiltonian, choosing arbitrary phases.
H2_sysbath = (1j*np.sqrt(lw_ext)*( (a_sys & a_ext.dag()) - (a_sys.dag() & a_ext))
             + np.sqrt(lw_int)*( (a_sys & a_int.dag()) + (a_sys.dag() & a_int)))

# Since only a_ext is monitored, we specify that the monitored/measured mode is mode "1" since it is the first in
#  the tensor product of the bath modes:
qg.measurement_rate(H1_disp, H2_sysbath, input2A_bath_state, meas_mode=1, qubit='e,g')[0]

np.float64(11.246644731114914)

In [7]:
# We notice that the measurement rate is lower than before due to inefficiencies in the monitoring and due to bath being in
# a thermal state. If we do not specify which bath mode(s) to measure, all are included, in which case the measurement rate 
# is the same as in the inital example if the input state is the vacuum:
input2B_bath_state = qg.vacuum(2)
qg.measurement_rate(H1_disp, H2_sysbath, input2B_bath_state, qubit='e,g')[0]

np.float64(11.909604841580638)

In [ ]:
"""
Now for something a bit different, we have two system modes which are not coherently coupled to each other, but are coupled to the
same qubit as well as the same bath. Additionally, squeezing is included on one of the modes to demonstrate the scope of the module.
"""
# Define the system and bath operators
a_sys = qg.tensor(qg.destroy(),qg.identity_cvs())
b_sys = qg.tensor(qg.identity_cvs(),qg.destroy())
sz_sys = qg.sigmaz()
s0_sys = qg.identity_fls(2)
c_bath = qg.destroy()

# In this case we will not include a drive term in the system Hamiltonian, but will instead model the drive as a displaced bath state.
input3_bath_state = qg.displaced(6)

# System parameters
chi_a = 5; chi_b = 3
lw_a = 12; lw_b = 20
amp = 2*np.exp(-0.5j*np.pi)
sqz = 2*np.exp(0.5j*np.pi)

# Define the system Hamiltonian for both phases
H3A_disp = ( (chi_a*a_sys.dag()*a_sys & sz_sys)
            + (chi_b*b_sys.dag()*b_sys & sz_sys)
            + ((amp*a_sys.dag()**2 + np.conj(amp)*a_sys**2) & s0_sys) 
           )
H3B_disp = ( (chi_a*a_sys.dag()*a_sys & sz_sys)
            + (chi_b*b_sys.dag()*b_sys & sz_sys)
            + ((sqz*a_sys.dag()**2 + np.conj(sqz)*a_sys**2) & s0_sys)
           )

# And now the bath Hamiltonian.
H3_sysbath = (1j*np.sqrt(lw_a)*( (a_sys & c_bath.dag()) - (a_sys.dag() & c_bath))
             + np.sqrt(lw_b)*( (b_sys & c_bath.dag()) + (b_sys.dag() & c_bath)))

In [9]:
# Note that due to the presence of the single-mode squeezing, that the noise component is amplified and so above vacuum...
qg.measurement_rate(H3A_disp, H3_sysbath, input3_bath_state, qubit='e,g')

(np.float64(3.3439767779390377),
 np.float64(4.35538752362948),
 np.float64(1.3024574669187132))

In [10]:
# ...but, if we change the phase then the noise is squeezed and hence below vacuum
qg.measurement_rate(H3B_disp, H3_sysbath, input3_bath_state, qubit='e,g')

(np.float64(5.620060493706709),
 np.float64(4.355387523629493),
 np.float64(0.7749716446124761))

In [2]:
"""
For multiple qubits, multiple measurement rates can be defined between each pair. Again, the pair of states to be distinguished can
be specified, or not, in which case all are calculated.
"""
# Operators
c_sys = qg.destroy()
c_bath = qg.destroy()
sz1 = qg.tensor(qg.sigmaz(),qg.identity_fls(2))
sz2 = qg.tensor(qg.identity_fls(2),qg.sigmaz())
s0 = qg.identity_fls([2,2])

# Environment is a displaced state
input4_bath_state = qg.displaced(5)

# Parameters
chi_c1 = 2.5; chi_c2 = 5; lw_c = 6

# We now define our system and system-bath operators
H4_disp = (chi_c1*c_sys.dag()*c_sys & sz1) + (chi_c2*c_sys.dag()*c_sys & sz2)
H4_sysbath = 1j*np.sqrt(lw_c)*( (c_sys & c_bath.dag()) - (c_sys.dag() & c_bath))

# Here, we will define a pair of trial measurement operators, the position and momentum quadratures
meas_oper_q = qg.position()
meas_oper_p = qg.momentum()

In [3]:
# We first check the measurement rate using the position quadrature operator...
qg.measurement_rate(H4_disp, H4_sysbath, input4_bath_state, meas_oper = meas_oper_q)[0]

array([[0.        , 5.11286489, 5.11286489, 0.        ],
       [5.11286489, 0.        , 0.        , 5.11286489],
       [5.11286489, 0.        , 0.        , 5.11286489],
       [0.        , 5.11286489, 5.11286489, 0.        ]])

In [4]:
# ...and notice that, compared to the momentum operator, it fails to distinguish certain pairs of pointer states.
qg.measurement_rate(H4_disp, H4_sysbath, input4_bath_state, meas_oper = meas_oper_p)[0]

array([[ 0.        , 17.4987801 ,  0.54004635, 11.89060642],
       [17.4987801 ,  0.        , 24.18704649,  0.54004635],
       [ 0.54004635, 24.18704649,  0.        , 17.4987801 ],
       [11.89060642,  0.54004635, 17.4987801 ,  0.        ]])

In [5]:
# Letting the function calculate the optimum measurement operator for each possible pair of pointer states, we can see that
# neither the position or momentum quadratures were optimal for every pair.
qg.measurement_rate(H4_disp, H4_sysbath, input4_bath_state)[0]

array([[ 0.        , 22.611645  ,  5.65291125, 11.89060642],
       [22.611645  ,  0.        , 24.18704649,  5.65291125],
       [ 5.65291125, 24.18704649,  0.        , 22.611645  ],
       [11.89060642,  5.65291125, 22.611645  ,  0.        ]])